In [1]:
import pandas as pd
import plotly.express as px
import dash
from dash import dcc, html
from dash.dependencies import Input, Output

In [2]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Historical_Wildfires.csv"
df = pd.read_csv(URL)

df['Month'] = pd.to_datetime(df['Date']).dt.month_name()
df['Year'] = pd.to_datetime(df['Date']).dt.year

app = dash.Dash(__name__)
app.config.suppress_callback_exceptions = True

In [3]:
title = html.H1('Australia Wildfire Dashboard', style={'textAlign': 'center', 'color': '#E46E6E', 'font-size': 36})

controls = html.Div([
    html.H2('Select Region:', style={'margin-right': '2em'}),
    dcc.RadioItems(
        options=[
            {'label': 'New South Wales', 'value': 'NSW'},
            {'label': 'Queensland', 'value': 'QL'},
            {'label': 'South Australia', 'value': 'SA'},
            {'label': 'Tasmania', 'value': 'TA'},
            {'label': 'Victoria', 'value': 'VI'},
            {'label': 'Western Australia', 'value': 'WA'},
            {'label': 'Northern Territory', 'value': 'NT'}
        ],
        value='NSW',
        id='region-id',
        inline=True
    ),
    html.H2('Select Year:', style={'margin-right': '2em'}),
    dcc.Dropdown(
        options=[{'label': year, 'value': year} for year in sorted(df['Year'].unique())],
        value=2005,
        id='year-id'
    )
])

In [4]:
app.layout = html.Div([
    title,
    controls,
    html.Div([
        html.Div(id='plot1', style={'width': '50%'}),
        html.Div(id='plot2', style={'width': '50%'})
    ], style={'display': 'flex'})
])

In [5]:
def create_pie(data, year):
    plot_data = data.groupby('Month')['Estimated_fire_area'].mean().reset_index()
    fig = px.pie(plot_data, values='Estimated_fire_area', names='Month', title=f'Monthly Average Estimated Fire Area in year {year}')
    return fig

In [6]:
def create_bar(data, year):
    plot_data = data.groupby('Month')['Count'].mean().reset_index()
    fig = px.bar(plot_data, x='Month', y='Count', title=f'Average Count of Pixels for Presumed Vegetation Fires in year {year}')
    return fig

In [7]:
@app.callback(
    [Output(component_id='plot1', component_property='children'),
     Output(component_id='plot2', component_property='children')],
    [Input(component_id='region-id', component_property='value'),
     Input(component_id='year-id', component_property='value')]
)
def update_graphs(selected_region, selected_year):
    filtered_df = df[(df['Region'] == selected_region) & (df['Year'] == int(selected_year))]
    
    fig1 = create_pie(filtered_df, selected_year)
    fig2 = create_bar(filtered_df, selected_year)
    
    return dcc.Graph(figure=fig1), dcc.Graph(figure=fig2)

In [8]:
app.run(debug=False, use_reloader=False, jupyter_mode="tab", port=8058)

Dash app running on http://127.0.0.1:8058/


<IPython.core.display.Javascript object>